In [0]:

PROJECT_NAME = "data_rewind"

print("\nCurrent catalog:")
spark.sql("SELECT current_catalog()").show(truncate=False)

print("\nCurrent schema:")
spark.sql("SELECT current_schema()").show(truncate=False)

In [0]:


orders_data = [
    (1001, 501, 200.00, "2026-10-01", "PLACED"),
    (1002, 502, 350.00, "2026-10-01", "PLACED"),
    (1003, 503, 150.00, "2026-10-02", "PLACED"),
    (1004, 501, 500.00, "2026-10-02", "PLACED"),
    (1005, 504, 250.00, "2026-10-03", "PLACED"),
]

customers_data = [
    (501, "Customer A", "Indore"),
    (502, "Customer B", "Bhopal"),
    (503, "Customer C", "Pune"),
    (504, "Customer D", "Mumbai"),
]

products_data = [
    (201, "Keyboard", "Accessories"),
    (202, "Mouse", "Accessories"),
    (203, "Monitor", "Electronics"),
]

print("Source records defined.")
print("Orders:", len(orders_data))
print("Customers:", len(customers_data))
print("Products:", len(products_data))

In [0]:
order_columns = [
    "order_id",
    "customer_id",
    "amount",
    "order_date",
    "status",
]

customer_columns = [
    "customer_id",
    "customer_name",
    "city",
]

products_columns=[
    "product_id",
    "product_name",
    "category"
]

orders_df = spark.createDataFrame(orders_data,order_columns)
customers_df= spark.createDataFrame(customers_data,customer_columns)
products_df = spark.createDataFrame(products_data,products_columns)


display(orders_df)
display(customers_df)
display(products_df)

In [0]:
from pyspark.sql import functions as F

bronze_orders = (
    orders_df
    .withColumn("source_system", F.lit("ecommerce"))
    .withColumn("ingested_at", F.current_timestamp())
)

bronze_customers = (
    customers_df
    .withColumn("source_system", F.lit("ecommerce"))
    .withColumn("ingested_at", F.current_timestamp())
)

bronze_products = (
    products_df
    .withColumn("source_system", F.lit("ecommerce"))
    .withColumn("ingested_at", F.current_timestamp())
)


bronze_orders.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "bronze_orders"
)

bronze_customers.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "bronze_customers"
)

bronze_products.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "bronze_products"
)

print("Bronze tables created successfully.")

In [0]:
for table_name in [
    "bronze_orders",
    "bronze_customers",
    "bronze_products",
]:
    table_df = spark.table(table_name)

    print(f"\n{table_name}: {table_df.count()} rows")
    display(table_df)

In [0]:
display(
    spark.sql("DESCRIBE HISTORY bronze_orders")
)

In [0]:
from pyspark.sql import functions as F

orders = spark.table("bronze_orders")

order_counts = (
    orders
    .groupBy("order_id")
    .count()
    .withColumnRenamed("count", "id_count")
)

checked_orders = orders.join(order_counts, on="order_id", how="left")


checked_orders = checked_orders.withColumn(
    "rejection_reason",
    F.concat_ws(
        ", ",
        F.when(F.col("order_id").isNull(), F.lit("missing_order_id")),
        F.when(F.col("amount").isNull(), F.lit("missing_amount")),
        F.when(F.col("amount") < 0, F.lit("negative_amount")),
        F.when(F.col("id_count") > 1, F.lit("duplicate_order_id"))
    )
)


silver_orders = checked_orders.filter(
    F.col("rejection_reason") == ""
).drop("id_count", "rejection_reason")

rejected_orders = checked_orders.filter(
    F.col("rejection_reason") != ""
)

print("Accepted rows:", silver_orders.count())
print("Rejected rows:", rejected_orders.count())

display(rejected_orders.select(
    "order_id", "amount", "rejection_reason"
))

In [0]:
silver_orders.write.format("delta").mode("overwrite").saveAsTable(
    "silver_orders"
)

rejected_orders.write.format("delta").mode("overwrite").saveAsTable(
    "rejected_orders"
)

print("Silver and rejected tables saved.")

In [0]:
print("Bronze:", spark.table("bronze_orders").count())
print("Silver:", spark.table("silver_orders").count())
print("Rejected:", spark.table("rejected_orders").count())

In [0]:
from pyspark.sql import functions as F

silver = spark.table("silver_orders")

gold_revenue = (
    silver
    .withColumn("order_date", F.to_date("order_date"))
    .groupBy("order_date", "status")
    .agg(
        F.countDistinct("order_id").alias("valid_order_count"),
        F.sum("amount").alias("total_revenue")
    )
    .withColumn("processed_at", F.current_timestamp())
)

gold_revenue.write.format("delta").mode("overwrite").saveAsTable(
    "gold_daily_revenue"
)

display(spark.table("gold_daily_revenue").orderBy("order_date"))

BASELINE FOR RECOVERY


In [0]:

baseline = (
    spark.table("gold_daily_revenue")
    .withColumn("baseline_captured_at", F.current_timestamp())
)

baseline.write.format("delta").mode("overwrite").saveAsTable(
    "rewind_gold_baseline"
)

print("Baseline captured.")
print("Baseline rows:", spark.table("rewind_gold_baseline").count())

display(spark.table("rewind_gold_baseline").orderBy("order_date"))

In [0]:
for table_name in [
    "bronze_orders",
    "silver_orders",
    "rejected_orders",
    "gold_daily_revenue",
    "rewind_gold_baseline",
]:
    print(f"{table_name}: {spark.table(table_name).count()} rows")

SIMULATING CORRUPTED PIPELINE RUN


In [0]:
from pyspark.sql import functions as F

original_orders = spark.table("bronze_orders")

duplicate_orders = original_orders.filter(
    F.col("order_id").isin(1001, 1002)
)

corrupted_orders = original_orders.unionByName(duplicate_orders)

corrupted_orders.write.format("delta").mode("overwrite").saveAsTable(
    "bronze_orders_incident"
)

print("Original rows:", original_orders.count())
print("Incident rows:", spark.table("bronze_orders_incident").count())

display(
    spark.table("bronze_orders_incident")
    .orderBy("order_id")
)

NOW IDENTIFYING THE INCIDENT

In [0]:
incident_source = spark.table("bronze_orders_incident")

duplicate_summary = (
    incident_source
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_summary.show()

duplicate_order_ids = [
    row["order_id"]
    for row in duplicate_summary.select("order_id").collect()
]

print("Duplicate order IDs:", duplicate_order_ids)
print("Number of duplicate IDs:", len(duplicate_order_ids))

CREATING DELTA LOGS

In [0]:
from datetime import datetime, timezone

incident_record = [(
    "INC-001",
    "bronze_orders_incident",
    "DUPLICATE_ORDER_IDS",
    len(duplicate_order_ids),
    "DETECTED",
    datetime.now(timezone.utc).replace(tzinfo=None)
)]

incident_columns = [
    "incident_id",
    "source_table",
    "incident_type",
    "affected_key_count",
    "status",
    "detected_at"
]

incident_df = spark.createDataFrame(
    incident_record,
    incident_columns
)

incident_df.write.format("delta").mode("append").saveAsTable(
    "rewind_incidents"
)

display(spark.table("rewind_incidents"))

QUANTIFYING THE IMPACT

In [0]:
from pyspark.sql import functions as F

incident_source = spark.table("bronze_orders_incident")


order_counts = (
    incident_source
    .groupBy("order_id")
    .count()
    .withColumnRenamed("count", "id_count")
)

checked = incident_source.join(
    order_counts, on="order_id", how="left"
)

checked = checked.withColumn(
    "rejection_reason",
    F.concat_ws(
        ", ",
        F.when(F.col("order_id").isNull(), "missing_order_id"),
        F.when(F.col("amount").isNull(), "missing_amount"),
        F.when(F.col("amount") < 0, "negative_amount"),
        F.when(F.col("id_count") > 1, "duplicate_order_id")
    )
)


incident_silver = checked.filter(
    F.col("rejection_reason") == ""
).drop("id_count", "rejection_reason")

incident_rejected = checked.filter(
    F.col("rejection_reason") != ""
)

incident_silver.write.format("delta").mode("overwrite").saveAsTable(
    "silver_orders_incident"
)

incident_rejected.write.format("delta").mode("overwrite").saveAsTable(
    "rejected_orders_incident"
)

print("Incident source:", incident_source.count())
print("Accepted:", incident_silver.count())
print("Rejected:", incident_rejected.count())

display(
    incident_rejected.select(
        "order_id", "amount", "rejection_reason"
    ).orderBy("order_id")
)

MEASURING DAMAGE

In [0]:
incident_gold = (
    spark.table("bronze_orders_incident")
    .withColumn("order_date", F.to_date("order_date"))
    .groupBy("order_date", "status")
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum("amount").alias("total_revenue")
    )
)

incident_gold.write.format("delta").mode("overwrite").saveAsTable(
    "gold_daily_revenue_incident"
)

baseline = spark.table("rewind_gold_baseline")
faulty = spark.table("gold_daily_revenue_incident")

comparison = (
    baseline.alias("b")
    .join(
        faulty.alias("f"),
        on=["order_date", "status"],
        how="full_outer"
    )
    .select(
        "order_date",
        "status",
        F.col("b.total_revenue").alias("baseline_revenue"),
        F.col("f.total_revenue").alias("incident_revenue"),
        (
            F.coalesce(F.col("f.total_revenue"), F.lit(0))
            - F.coalesce(F.col("b.total_revenue"), F.lit(0))
        ).alias("revenue_difference")
    )
    .orderBy("order_date")
)

display(comparison)

VERIFYING IMPACT

In [0]:
print("Baseline revenue:")
spark.table("rewind_gold_baseline").orderBy("order_date").show()

print("Incident revenue:")
spark.table("gold_daily_revenue_incident").orderBy("order_date").show()

print("Incident records rejected:")
spark.table("rejected_orders_incident").count()

RESTORING BASELINE

In [0]:
from pyspark.sql import functions as F


baseline = spark.table("rewind_gold_baseline")

(
    baseline
    .drop("baseline_captured_at")
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_daily_revenue")
)

print("Gold table restored from baseline.")
display(
    spark.table("gold_daily_revenue").orderBy("order_date")
)

VERIFYING RECOVERY

In [0]:
baseline = spark.table("rewind_gold_baseline")
restored = spark.table("gold_daily_revenue")

differences = (
    baseline.alias("b")
    .join(
        restored.alias("r"),
        on=["order_date", "status"],
        how="full_outer"
    )
    .filter(
        ~F.col("b.total_revenue").eqNullSafe(
            F.col("r.total_revenue")
        )
        | ~F.col("b.valid_order_count").eqNullSafe(
            F.col("r.valid_order_count")
        )
    )
)

difference_count = differences.count()

print("Mismatched rows:", difference_count)
print("Recovery verified:", difference_count == 0)

display(restored.orderBy("order_date"))

INSPECT DELTA HISTORY

In [0]:
display(
    spark.sql("DESCRIBE HISTORY gold_daily_revenue")
)

Data Rewind — Pipeline Controller

In [0]:
from datetime import datetime, timezone
from pyspark.sql import functions as F

def table_exists(table_name):
    return spark.catalog.tableExists(table_name)

def get_row_count(table_name):
    return spark.table(table_name).count()

def get_latest_delta_version(table_name):
    history = spark.sql(f"DESCRIBE HISTORY {table_name}")
    return history.orderBy(F.col("version").desc()).first()["version"]

pipeline_run_id = (
    datetime.now(timezone.utc)
    .strftime("run_%Y%m%dT%H%M%SZ")
)

print("Pipeline run:", pipeline_run_id)

for table_name in [
    "bronze_orders",
    "silver_orders",
    "rejected_orders",
    "gold_daily_revenue",
    "rewind_gold_baseline",
]:
    exists = table_exists(table_name)

    print(
        table_name,
        "| exists:", exists,
        "| rows:", get_row_count(table_name) if exists else "N/A"
    )

print(
    "Current Gold version:",
    get_latest_delta_version("gold_daily_revenue")
)

AUDIT TABLES

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    LongType,
)


run_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("pipeline_name", StringType(), False),
    StructField("status", StringType(), False),
    StructField("started_at", TimestampType(), False),
    StructField("finished_at", TimestampType(), True),
    StructField("error_message", StringType(), True),
])


incident_schema = StructType([
    StructField("incident_id", StringType(), False),
    StructField("run_id", StringType(), False),
    StructField("incident_type", StringType(), False),
    StructField("source_table", StringType(), False),
    StructField("affected_key_count", LongType(), False),
    StructField("status", StringType(), False),
    StructField("detected_at", TimestampType(), False),
    StructField("recovery_version", LongType(), True),
    StructField("verification_passed", StringType(), True),
])


spark.createDataFrame([], run_schema) \
    .write.format("delta").mode("ignore").saveAsTable("rewind_pipeline_runs")

spark.createDataFrame([], incident_schema) \
    .write.format("delta").mode("ignore").saveAsTable("rewind_incident_audit")

print("Audit tables are ready.")

display(spark.table("rewind_pipeline_runs"))
display(spark.table("rewind_incident_audit"))

RECORDING PIPELINE RUNS

In [0]:
from datetime import datetime, timezone
from uuid import uuid4

def record_pipeline_run(
    run_id,
    status,
    started_at,
    finished_at=None,
    error_message=None,
):
    record = [(
        run_id,
        "data_rewind",
        status,
        started_at,
        finished_at,
        error_message,
    )]

    run_df = spark.createDataFrame(
        record,
        schema=run_schema,
    )

    run_df.write.format("delta").mode("append").saveAsTable(
        "rewind_pipeline_runs"
    )


run_id = f"run_{uuid4().hex[:12]}"
started_at = datetime.now(timezone.utc).replace(tzinfo=None)
YES 
finished_at = datetime.now(timezone.utc).replace(tzinfo=None)

record_pipeline_run(
    run_id=run_id,
    status="SUCCEEDED",
    started_at=started_at,
    finished_at=finished_at,
)

print("Recorded run:", run_id)

display(
    spark.table("rewind_pipeline_runs")
    .orderBy("started_at", ascending=False)
)

AUDIT FUNCTION

In [0]:
from datetime import datetime, timezone
from uuid import uuid4

def record_incident(
    run_id,
    incident_type,
    source_table,
    affected_key_count,
    status="DETECTED",
    recovery_version=None,
    verification_passed=None,
):
    incident_record = [(
        f"inc_{uuid4().hex[:12]}",
        run_id,
        incident_type,
        source_table,
        int(affected_key_count),
        status,
        datetime.now(timezone.utc).replace(tzinfo=None),
        recovery_version,
        (
            str(verification_passed).lower()
            if verification_passed is not None
            else None
        ),
    )]

    incident_df = spark.createDataFrame(
        incident_record,
        schema=incident_schema,
    )

    incident_df.write.format("delta").mode("append").saveAsTable(
        "rewind_incident_audit"
    )

    return incident_record[0][0]


test_incident_id = record_incident(
    run_id=run_id,
    incident_type="DUPLICATE_ORDER_IDS",
    source_table="bronze_orders_incident",
    affected_key_count=2,
)

print("Recorded incident:", test_incident_id)

display(
    spark.table("rewind_incident_audit")
    .orderBy("detected_at", ascending=False)
)

AUTOMATE INCIDENT DETECTION

In [0]:
from uuid import uuid4
from pyspark.sql import functions as F

def detect_duplicate_incident(source_table, run_id):
    source = spark.table(source_table)

    duplicates = (
        source
        .groupBy("order_id")
        .count()
        .filter(F.col("count") > 1)
    )

    affected_key_count = duplicates.count()

    if affected_key_count == 0:
        print(f"No duplicate IDs found in {source_table}.")
        return None

    incident_id = record_incident(
        run_id=run_id,
        incident_type="DUPLICATE_ORDER_IDS",
        source_table=source_table,
        affected_key_count=affected_key_count,
        status="DETECTED",
    )

    print("Incident detected:", incident_id)
    print("Affected order IDs:", affected_key_count)

    display(duplicates)

    return incident_id


incident_run_id = f"run_{uuid4().hex[:12]}"

incident_id = detect_duplicate_incident(
    source_table="bronze_orders_incident",
    run_id=incident_run_id,
)

In [0]:
display(
    spark.table("rewind_incident_audit")
    .orderBy("detected_at", ascending=False)
)

ADDING RECOVERY FUNCTION

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

def recover_gold(incident_id):
    
    baseline = spark.table("rewind_gold_baseline")

    
    restored_data = baseline.drop("baseline_captured_at")

    (
        restored_data.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("gold_daily_revenue")
    )

    
    restored = spark.table("gold_daily_revenue")

   
    keys = ["order_date", "status"]

    comparison = (
        baseline.alias("b")
        .join(restored.alias("r"), on=keys, how="full_outer")
        .filter(
            ~F.col("b.total_revenue").eqNullSafe(
                F.col("r.total_revenue")
            )
            | ~F.col("b.valid_order_count").eqNullSafe(
                F.col("r.valid_order_count")
            )
        )
    )

    mismatch_count = comparison.count()
    verified = mismatch_count == 0

    
    recovery_version = (
        spark.sql("DESCRIBE HISTORY gold_daily_revenue")
        .orderBy(F.col("version").desc())
        .first()["version"]
    )

   
    audit = DeltaTable.forName(spark, "rewind_incident_audit")

    audit.update(
        condition=F.col("incident_id") == incident_id,
        set={
            "status": F.lit(
                "VERIFIED" if verified else "RECOVERY_FAILED"
            ),
            "recovery_version": F.lit(int(recovery_version)),
            "verification_passed": F.lit(str(verified).lower()),
        },
    )

    print("Incident:", incident_id)
    print("Recovery version:", recovery_version)
    print("Mismatched rows:", mismatch_count)
    print("Recovery verified:", verified)

    return verified

EXECUTE

In [0]:
recovery_success = recover_gold(incident_id)

display(
    spark.table("rewind_incident_audit")
    .filter(F.col("incident_id") == incident_id)
)

VERIFY

In [0]:
display(
    spark.table("gold_daily_revenue")
    .orderBy("order_date")
)

display(
    spark.sql("DESCRIBE HISTORY gold_daily_revenue")
)

In [0]:
display(
    spark.table("rewind_incident_audit")
    .orderBy("detected_at", ascending=False)
)

In [0]:
from src.data_rewind.quality import validate_orders

test_valid, test_rejected = validate_orders(
    spark.table("bronze_orders_incident")
)

print("Valid rows:", test_valid.count())
print("Rejected rows:", test_rejected.count())

display(test_rejected)